In [1]:
import numpy as np
import pandas as pd

# =========================================================
# CPS MODEL: Awareness-triggered Event Migration + SIR
# 核心函数部分（完全不变）
# =========================================================

def compute_epsilon(states, alpha, epsilon0):
    p_A = states[:, 1] + states[:, 2] + states[:, 4]
    return np.where(p_A > alpha, epsilon0, 0.0)

def compute_effective_awareness(n, states, g_i, R):
    N, M = R.shape
    p_A = states[:, 1] + states[:, 2] + states[:, 4]
    tilde_p_A = np.zeros(M)
    for j in range(M):
        numerator, denominator = 0.0, 0.0
        for i in range(N):
            flow = n[i] * g_i[i] * R[i, j]
            numerator += flow * p_A[i]
            denominator += flow
        tilde_p_A[j] = numerator / denominator if denominator > 0 else 0
    return tilde_p_A

def compute_awareness_conversion(R, lambda1, tilde_p_A):
    N, M = R.shape
    r = np.zeros(N)
    for i in range(N):
        prod = 1.0
        for j in range(M):
            prod *= (1 - lambda1 * R[i, j] * tilde_p_A[j])
        r[i] = 1 - prod
    return r

def compute_infection_residence(n, states, g_i, beta_U, beta_A):
    n_remain = n * (1 - g_i)
    AI = states[:, 2]
    N_U = 1 - np.power(1 - beta_U * AI, n_remain)
    N_A = 1 - np.power(1 - beta_A * AI, n_remain)
    return N_U, N_A

def compute_infection_transit(n, states, g_i, R, beta_U, beta_A):
    N, M = R.shape
    AI = states[:, 2]
    M_U = np.zeros(M)
    M_A = np.zeros(M)
    for j in range(M):
        prod_U, prod_A = 1.0, 1.0
        for k in range(N):
            n_to_j = n[k] * g_i[k] * R[k, j]
            prod_U *= np.power(1 - beta_U * AI[k], n_to_j)
            prod_A *= np.power(1 - beta_A * AI[k], n_to_j)
        M_U[j] = 1 - prod_U
        M_A[j] = 1 - prod_A
    return M_U, M_A

def compute_overall_infection(g_i, R, N_U, N_A, M_U, M_A):
    Q_U = (1 - g_i) * N_U + g_i * np.dot(R, M_U)
    Q_A = (1 - g_i) * N_A + g_i * np.dot(R, M_A)
    return Q_U, Q_A

def update_states(states, r, Q_U, Q_A, mu1, mu2):
    US, AS, AI, UR, AR = states.T
    new_US = US * (1 - r) * (1 - Q_U) + AS * mu1 * (1 - Q_U)
    new_AS = AS * (1 - mu1) * (1 - Q_A) + US * r * (1 - Q_A)
    new_AI = (AI * (1 - mu2)
              + AS * (1 - mu1) * Q_A
              + AS * mu1 * Q_U
              + US * (1 - r) * Q_U
              + US * r * Q_A)
    new_UR = AI * mu1 * mu2 + AR * mu1 + UR * (1 - r)
    new_AR = AR * (1 - mu1) + AI * (1 - mu1) * mu2 + UR * r
    new_states = np.vstack([new_US, new_AS, new_AI, new_UR, new_AR]).T
    new_states /= new_states.sum(axis=1, keepdims=True)
    return new_states

def update_population(n, states, g_i, epsilon, R, T):
    N, M = R.shape
    X = states * n[:, None]
    X_remain = X - (X.T * (g_i * epsilon)).T
    m_X = np.zeros((M, 5))
    for j in range(M):
        for i in range(N):
            flow = X[i] * g_i[i] * epsilon[i] * R[i, j]
            m_X[j] += flow
    flow_return_X = np.zeros((N, 5))
    for j in range(M):
        for i in range(N):
            flow_return_X[i] += m_X[j] * T[i, j]
    X_new = X_remain + flow_return_X
    n_new = np.sum(X_new, axis=1)
    states_new = X_new / n_new[:, None]
    return n_new, states_new

def main_simulation(Time, N, M, params):
    lambda1 = params["lambda1"]
    mu1 = params["mu1"]
    mu2 = params["mu2"]
    beta_U = params["beta_U"]
    beta_A = params["beta_A"]
    g = params["g"]
    theta = params["theta"]
    epsilon0 = params["epsilon0"]
    alpha = params["alpha"]

    g_i = np.full(N, theta * g)
    n = np.full(N, 200.0)

    states = np.zeros((N, 5))
    states[:, 0] = 0.99
    states[:, 2] = 0.01

    w = np.random.rand(N, M)
    Rmat = w / w.sum(axis=1, keepdims=True)
    Tmat = w / w.sum(axis=0, keepdims=True)

    history = np.zeros((Time, 5))

    for t in range(Time):
        global_state = np.sum(states * n[:, None], axis=0) / np.sum(n)
        history[t] = global_state

        epsilon = compute_epsilon(states, alpha, epsilon0)
        n_new, states_new = update_population(n, states, g_i, epsilon, Rmat, Tmat)
        tilde_p_A = compute_effective_awareness(n_new, states_new, g_i, Rmat)
        r = compute_awareness_conversion(Rmat, lambda1, tilde_p_A)
        N_U, N_A = compute_infection_residence(n_new, states_new, g_i, beta_U, beta_A)
        M_U, M_A = compute_infection_transit(n_new, states_new, g_i, Rmat, beta_U, beta_A)
        Q_U, Q_A = compute_overall_infection(g_i, Rmat, N_U, N_A, M_U, M_A)
        states = update_states(states_new, r, Q_U, Q_A, mu1, mu2)
        n = n_new

    return history


# =========================================================
# 扫描 λ 并计算峰值感染密度的重复实验
# =========================================================
if __name__ == "__main__":
    Time = 150
    N = 15
    M = 5
    num_simulations = 2          # 每个 λ 重复实验次数
    base_seed = 12345             # 基础随机种子，用于可重复性

    # 固定参数（λ 将被循环覆盖）
    base_params = {
        "lambda1": None,          # 占位，将在循环中设置
        "mu1": 0.15,
        "mu2": 0.1,
        "beta_U": 0.003,
        "beta_A": 0.0015,
        "g": 0.2,
        "epsilon0": 0.5,
        "alpha": 0.5,
        "theta": 1.0
    }

    # 定义要扫描的 λ 值
    lambda_list = np.linspace(0.4, 0.8, 81)

    # 存储结果
    results = []

    for lambda1 in lambda_list:

        # 更新当前 λ
        params = base_params.copy()
        params["lambda1"] = lambda1

        # 记录本次 λ 的所有峰值
        peak_list = []

        for sim in range(num_simulations):
            # 为每次实验设置独立的随机种子（保证可复现）
            seed = base_seed + 1000
            np.random.seed(seed)

            # 运行仿真
            history = main_simulation(Time, N, M, params)

            # 提取感染比例 AI（全局均值）的时间序列
            ai_global = history[:, 2]   # 第2列是 AI
            peak_infection = np.max(ai_global)
            peak_list.append(peak_infection)


        # 计算均值
        peak_mean = np.mean(peak_list)

        results.append({
            "lambda": lambda1,
            "peak_mean": peak_mean,
        })


    # 保存结果到 CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv("CPS_peak_vs_lambda.csv", index=False)
    print("\n所有 λ 扫描完成，结果已保存至 CPS_peak_vs_lambda.csv")
    print(df_results.head())


所有 λ 扫描完成，结果已保存至 CPS_peak_vs_lambda.csv
   lambda  peak_mean
0   0.400   0.344163
1   0.405   0.343133
2   0.410   0.342107
3   0.415   0.341084
4   0.420   0.340065
